# Fill a missing AdmR value: one drug

This is the first, deliberately small run of deliverable D3: fill a `NaN` in the `AdmR` column of `inputs/mode_of_administration.md` for **one drug and one ATC code**, using DailyMed label counts.

Rules applied (from the task spec):

- **Rule 3**: at most 2 routes per ATC code, intravenous excluded.
- **Rule 4**: rank routes by how many DailyMed labels report them for this ATC code.
- **Rule 5**: one output row per route.
- **Rule 7**: every label used is saved to `results/` with its DailyMed link and a timestamp; the raw API response is kept in `evidence/dailymed/`.

The source table is never edited in place; results go to a new CSV.

In [1]:
import json, re, datetime
from collections import Counter
from pathlib import Path

import pandas as pd
import requests

DRUG = "azelastine"
ATC = "S01GX07"

HERE = Path.cwd() if (Path.cwd() / "inputs").exists() else Path.cwd() / "curation"
INPUT = HERE / "inputs" / "mode_of_administration.md"
LEGEND = HERE / "inputs" / "fda_route_of_administration.tsv"
WHO_MAP = HERE / "inputs" / "who_to_fda_route.csv"
CACHE = HERE / "evidence" / "dailymed" / f"{DRUG}_spls.json"
RESULTS = HERE / "results"
RESULTS.mkdir(exist_ok=True)

## 1. Load the table and find the row

In [2]:
rows = [l.strip().strip("|").split(" | ") for l in INPUT.read_text().splitlines() if l.startswith("|")]
rows = [[c.strip() for c in r] for r in rows if len(r) >= 4]
admr = pd.DataFrame(rows[2:], columns=rows[0])
print(len(admr), "rows,", (admr.AdmR == "NaN").sum(), "with AdmR = NaN")
admr[admr.GenericName == DRUG]

1167 rows, 315 with AdmR = NaN


,CID,GenericName,ATC,AdmR
11,2267,azelastine,R06AX19,O
12,2267,azelastine,S01GX07,NaN
13,2267,azelastine,R01AC03,N


## 1b. Route legend: FDA Route of Administration

`AdmR` now uses the FDA Route of Administration legend (112 routes, each with a definition, short name, FDA code and NCI concept ID), stored verbatim in `inputs/fda_route_of_administration.tsv`. These are the same route terms DailyMed labels use, so a label's route can be recorded without translation. It covers the specific routes the coarse WHO codes cannot, such as INTRATHECAL (103), EPIDURAL (009), INTRAVITREAL (311) and OPHTHALMIC (012).

Points found when checking each row of the legend:

- `EXTRA‑AMNIOTIC` uses a non-breaking hyphen (U+2011), so a plain-hyphen search will not find it.
- INTRATHECAL's definition includes injection into the cerebral ventricles, which overlaps INTRAVENTRICULAR.
- Rule 3 (exclude IV) has to drop all three intravenous entries: INTRAVENOUS, INTRAVENOUS BOLUS and INTRAVENOUS DRIP.
- INTRACAMERAL (used in the Routes of Drug Administration article, e.g. cefuroxime after cataract surgery) is not in this legend; INTRAOCULAR is the closest entry.
- PARENTERAL, OTHER, UNASSIGNED, UNKNOWN and NOT APPLICABLE are catch-alls. They are used only when nothing more specific is known.

In [3]:
legend = pd.read_csv(LEGEND, sep="\t", dtype=str)
assert len(legend) == 112 and legend["FDA CODE"].is_unique and legend["NCI CONCEPT ID"].is_unique
legend.head()

,NAME,DEFINITION,SHORT NAME,FDA CODE,NCI CONCEPT ID
0,AURICULAR (OTIC),Administration to or by way of the ear.,OTIC,013,C38192
1,BUCCAL,"Administration directed toward the cheek, gene...",BUCCAL,030,C38193
2,CONJUNCTIVAL,"Administration to the conjunctiva, the delicat...",CONJUNC,068,C38194
3,CUTANEOUS,Administration to the skin.,CUTAN,130,C38675
4,DENTAL,Administration to a tooth or teeth.,DENTAL,038,C38197


The table's existing WHO codes mapped to FDA routes (`inputs/who_to_fda_route.csv`). `P` maps only to the coarse PARENTERAL; the specific injection route needs each drug's labels. Rows marked `check` are best guesses to confirm.

In [4]:
who_map = pd.read_csv(WHO_MAP, dtype=str).fillna("")
assert set(who_map.fda_route) <= set(legend.NAME)
print("existing codes with no mapping:", sorted(set(admr.AdmR) - set(who_map.who_admr) - {"NaN"}))
admr.merge(who_map, left_on="AdmR", right_on="who_admr", how="left") \
    .groupby(["AdmR", "fda_route", "confidence"], dropna=False).size().sort_values(ascending=False)

existing codes with no mapping: []


AdmR            fda_route                 confidence
O               ORAL                      exact         477
NaN             NaN                       NaN           315
P               PARENTERAL                coarse        237
R               RECTAL                    exact          26
N               NASAL                     exact          22
Inhal.powder    RESPIRATORY (INHALATION)  exact          19
Inhal.aerosol   RESPIRATORY (INHALATION)  exact          13
Inhal.solution  RESPIRATORY (INHALATION)  exact          13
TD              TRANSDERMAL               exact          11
SL              SUBLINGUAL                exact          10
V               VAGINAL                   exact          10
implant         PARENTERAL                coarse          4
Inhal           RESPIRATORY (INHALATION)  exact           3
Chewing gum     ORAL                      check           1
TD patch        TRANSDERMAL               exact           1
lamella         OPHTHALMIC                check

## 2. Get the DailyMed labels

Live call to the DailyMed SPL API. If the network blocks it, the notebook falls back to the saved response in `evidence/dailymed/`, so the run is reproducible offline.

In [5]:
def dailymed_spls(name):
    url = "https://dailymed.nlm.nih.gov/dailymed/services/v2/spls.json"
    out, page = [], 1
    while True:
        r = requests.get(url, params={"drug_name": name, "pagesize": 100, "page": page}, timeout=20)
        r.raise_for_status()
        j = r.json()
        out += [{"setid": d["setid"], "title": d["title"]} for d in j["data"]]
        if j["metadata"]["next_page"] in (None, "null"):
            return out

try:
    labels = dailymed_spls(DRUG)
    retrieved = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
    CACHE.write_text(json.dumps({"retrieved_utc": retrieved, "labels": labels}, indent=1))
    source = "live"
except Exception as e:
    cached = json.loads(CACHE.read_text())
    labels, retrieved, source = cached["labels"], cached["retrieved_utc"], f"cache ({type(e).__name__})"
print(len(labels), "labels from", source, "retrieved", retrieved)

47 labels from cache (ProxyError) retrieved 2026-10-09T03:37:06+00:00


## 3. Route of each label, and which ATC code it belongs to

Each label title is matched first against the 112 FDA route names (longest first, whole words, so SUBCONJUNCTIVAL is not read as CONJUNCTIVAL and INTRAVENOUS DRIP is not read as INTRAVENOUS). Only if no route is named does the dosage form decide (TABLET → ORAL, SPRAY → NASAL, INJECTION → PARENTERAL...). A drug can have several ATC codes (azelastine: R06AX19 oral, R01AC03 nasal, S01GX07 eye), so only labels whose route fits **this** ATC code are counted (rule 4 is per ATC).

In [6]:
SYNONYMS = {"AURICULAR (OTIC)": r"OTIC|AURICULAR|\bEAR\b", "RESPIRATORY (INHALATION)": r"INHAL",
            "OPHTHALMIC": r"OPHTHALMIC|\bEYE\b", "EXTRA‑AMNIOTIC": r"EXTRA[-‑]AMNIOTIC"}
CATCH_ALL = {"PARENTERAL", "OTHER", "UNASSIGNED", "UNKNOWN", "NOT APPLICABLE"}
FORM_TO_ROUTE = [(r"SOLUTION/ ?DROPS", "OPHTHALMIC"), (r"SPRAY", "NASAL"), (r"AEROSOL|NEBULI", "RESPIRATORY (INHALATION)"),
                 (r"PATCH", "TRANSDERMAL"), (r"SUPPOSITORY|ENEMA", "RECTAL"), (r"CREAM|OINTMENT|GEL|LOTION|FOAM", "TOPICAL"),
                 (r"TABLET|CAPSULE|SYRUP|ELIXIR|GRANULE", "ORAL"), (r"INJECTION|INJECTABLE", "PARENTERAL")]
named = sorted((n for n in legend.NAME if n not in CATCH_ALL), key=len, reverse=True)
matchers = [(n, SYNONYMS.get(n, r"\b" + re.escape(n) + r"\b")) for n in named] + [(r, p) for p, r in FORM_TO_ROUTE]
IV_ROUTES = {"INTRAVENOUS", "INTRAVENOUS BOLUS", "INTRAVENOUS DRIP"}
ATC_PREFIX_ROUTES = {"S01": {"OPHTHALMIC", "CONJUNCTIVAL", "INTRAOCULAR", "INTRAVITREAL", "SUBCONJUNCTIVAL"},
                     "S02": {"AURICULAR (OTIC)"}, "R01": {"NASAL"}, "R06": {"ORAL"}}

def route_of(title):
    for route, pattern in matchers:
        if re.search(pattern, title.upper()):
            return route
    return "UNKNOWN"

checks = {"LIORESAL INTRATHECAL (BACLOFEN) INJECTION": "INTRATHECAL", "MORPHINE SULFATE INJECTION": "PARENTERAL",
          "CIPROFLOXACIN OTIC SOLUTION/ DROPS": "AURICULAR (OTIC)", "FENTANYL BUCCAL TABLET": "BUCCAL",
          "TRIAMCINOLONE INJECTION, SUSPENSION INTRA-ARTICULAR": "INTRA-ARTICULAR",
          "DEXAMETHASONE SUBCONJUNCTIVAL INJECTION": "SUBCONJUNCTIVAL", "HEPARIN INTRAVENOUS DRIP": "INTRAVENOUS DRIP",
          "NICOTINE TRANSDERMAL SYSTEM PATCH": "TRANSDERMAL", "AZELASTINE HYDROCHLORIDE SPRAY, METERED": "NASAL"}
assert all(route_of(t) == want for t, want in checks.items()), {t: route_of(t) for t in checks}

assert set().union(*ATC_PREFIX_ROUTES.values()) <= set(legend.NAME)

lab = pd.DataFrame(labels)
lab["fda_route"] = lab.title.map(route_of)
allowed = ATC_PREFIX_ROUTES.get(ATC[:3])
lab["counts_for_this_ATC"] = lab.fda_route.isin(allowed) if allowed else True
lab.fda_route.value_counts()

fda_route
NASAL         38
OPHTHALMIC     9
Name: count, dtype: int64

## 4. Apply rules 3 and 4: drop IV (all three IV entries), keep the top 2

In [7]:
counts = Counter(lab.loc[lab.counts_for_this_ATC & ~lab.fda_route.isin(IV_ROUTES), "fda_route"])
top = counts.most_common(2)
print(f"{ATC}: route counts {dict(counts)} -> keep {top}")

S01GX07: route counts {'OPHTHALMIC': 9} -> keep [('OPHTHALMIC', 9)]


## 5. Rule 5: one row per route, then save

In [8]:
n_total = len(lab)
L = legend.set_index("NAME")
out = pd.DataFrame([{
    "CID": admr.loc[(admr.GenericName == DRUG) & (admr.ATC == ATC), "CID"].iloc[0],
    "GenericName": DRUG, "ATC": ATC, "AdmR_before": "NaN",
    "AdmR_after": route, "fda_short_name": L.at[route, "SHORT NAME"], "fda_code": L.at[route, "FDA CODE"],
    "nci_concept_id": L.at[route, "NCI CONCEPT ID"],
    "Edit_Flag": "filled_from_DailyMed",
    "evidence": f"{n} of {n_total} DailyMed labels for {DRUG} have this route",
} for route, n in top])
out.to_csv(RESULTS / f"admr_{DRUG}_{ATC}.csv", index=False)

prov = lab.assign(drug=DRUG, source_url="https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=" + lab.setid,
                  retrieved_utc=retrieved)
prov.to_csv(RESULTS / f"provenance_{DRUG}.csv", index=False)
out

,CID,GenericName,ATC,AdmR_before,AdmR_after,fda_short_name,fda_code,nci_concept_id,Edit_Flag,evidence
0,2267,azelastine,S01GX07,NaN,OPHTHALMIC,OPHTHALM,012,C38287,filled_from_DailyMed,9 of 47 DailyMed labels for azelastine have th...


## What is still open

- IV-only drugs such as alfentanil (N01AH02) have no route left after rule 3; how to record them is waiting on Star.
- The 237 existing `P` rows map only to PARENTERAL; the specific route (INTRAMUSCULAR, SUBCUTANEOUS, INTRATHECAL...) needs each drug's labels. Mapping all existing codes to FDA names is a separate step, not done in this notebook.
- Label titles often do not name the route (e.g. "MORPHINE SULFATE INJECTION"); reading the route field inside each SPL would remove that guess.
- PubChem, WHO ATC/DDD and DrugBank are blocked from the cloud session that ran this notebook, so only DailyMed was used.